# Steam Review 데이터 전처리

## Part 1: 수집된 데이터 확인

steam_train_12k.jsonl 파일의 검증 및 라벨링 결과 분포를 확인합니다.

**검증 항목:**
- JSONL 파일 파싱 및 구조 검증
- 필수 필드 확인: `id`, `sentence_form`, `annotation`
- Annotation 형식: `[속성명, 감정]` 쌍
- 속성명: 정의된 17개 속성만 포함 여부
- 감정: positive, negative, neutral 중 하나
- 중복: 리뷰 내 중복 annotation, 같은 ID 중복

In [2]:
from pathlib import Path
import json
from collections import Counter, defaultdict

try:
    import pandas as pd
except ImportError:
    pd = None

# 데이터 파일 경로 (현재 경로 기준)
DATA_FILE = Path("labelled_AI_final/steam_train_12k.jsonl")

# 정의된 17개 속성
STEAM_ASPECTS = [
    "최적화#프레임", "최적화#조작감", "시스템#버그",
    "콘텐츠#볼륨", "콘텐츠#몰입도", "콘텐츠#난이도", "콘텐츠#피로도",
    "시스템#밸런스", "시스템#독창성", "시스템#자유도", "시스템#진입장벽",
    "UX#그래픽", "UX#캐릭터디자인", "UX#사운드",
    "스토리#내러티브", "운영#핵/치트", "운영#업데이트"
]

VALID_ASPECTS = set(STEAM_ASPECTS)
VALID_SENTIMENTS = {"positive", "negative", "neutral"}
REQUIRED_FIELDS = {"id", "sentence_form", "annotation"}

print(f"데이터 파일: {DATA_FILE.resolve()}")
print(f"파일 존재: {DATA_FILE.exists()}")

c:\Users\WONHO\anaconda3\envs\intro_ai\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


데이터 파일: C:\Users\WONHO\Desktop\SSU\26-1\datascience\project\labelled_AI_final\steam_train_12k.jsonl
파일 존재: True


In [3]:
def add_issue(records, *, issue_type, line_no, item_id=None, message="", value=None, sentence_form=None, annotation=None):
    records.append({
        "issue_type": issue_type,
        "line_no": line_no,
        "id": item_id,
        "message": message,
        "value": value,
        "sentence_form": sentence_form,
        "annotation": annotation,
    })


def validate_jsonl(path):
    issues = []
    rows = []
    id_to_lines = defaultdict(list)
    aspect_counts = Counter()
    sentiment_counts = Counter()
    total_nonempty_lines = 0
    total_annotations = 0
    annotated_rows = 0

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue

            total_nonempty_lines += 1

            try:
                item = json.loads(line)
            except json.JSONDecodeError as e:
                add_issue(
                    issues,
                    issue_type="json_parse_error",
                    line_no=line_no,
                    message=str(e),
                    value=line[:300],
                )
                continue

            if not isinstance(item, dict):
                add_issue(
                    issues,
                    issue_type="row_not_object",
                    line_no=line_no,
                    message="각 JSONL 줄은 JSON 객체여야 합니다.",
                    value=repr(type(item)),
                )
                continue

            item_id = item.get("id")
            sentence_form = item.get("sentence_form")
            annotation = item.get("annotation")
            rows.append(item)

            if item_id is not None:
                id_to_lines[item_id].append(line_no)

            missing = REQUIRED_FIELDS - set(item)
            if missing:
                add_issue(
                    issues,
                    issue_type="missing_required_field",
                    line_no=line_no,
                    item_id=item_id,
                    message="필수 필드가 없습니다.",
                    value=sorted(missing),
                    sentence_form=sentence_form,
                    annotation=annotation,
                )

            if "id" in item and not isinstance(item_id, str):
                add_issue(issues, issue_type="id_not_string", line_no=line_no, item_id=item_id, value=repr(type(item_id)), sentence_form=sentence_form, annotation=annotation)

            if "sentence_form" in item and not isinstance(sentence_form, str):
                add_issue(issues, issue_type="sentence_form_not_string", line_no=line_no, item_id=item_id, value=repr(type(sentence_form)), sentence_form=sentence_form, annotation=annotation)

            if "annotation" in item and not isinstance(annotation, list):
                add_issue(issues, issue_type="annotation_not_list", line_no=line_no, item_id=item_id, value=repr(type(annotation)), sentence_form=sentence_form, annotation=annotation)
                continue

            if not isinstance(annotation, list):
                continue

            if annotation:
                annotated_rows += 1

            seen_in_row = set()
            for ann_idx, ann in enumerate(annotation):
                total_annotations += 1

                if not isinstance(ann, list):
                    add_issue(issues, issue_type="annotation_item_not_list", line_no=line_no, item_id=item_id, message=f"annotation[{ann_idx}]는 리스트여야 합니다.", value=ann, sentence_form=sentence_form, annotation=annotation)
                    continue

                if len(ann) != 2:
                    add_issue(issues, issue_type="annotation_item_len_not_2", line_no=line_no, item_id=item_id, message=f"annotation[{ann_idx}]는 [속성명, 감정]이어야 합니다.", value=ann, sentence_form=sentence_form, annotation=annotation)
                    continue

                aspect, sentiment = ann

                if not isinstance(aspect, str):
                    add_issue(issues, issue_type="aspect_not_string", line_no=line_no, item_id=item_id, message=f"annotation[{ann_idx}][0]은 문자열이어야 합니다.", value=repr(type(aspect)), sentence_form=sentence_form, annotation=annotation)
                    continue

                if not isinstance(sentiment, str):
                    add_issue(issues, issue_type="sentiment_not_string", line_no=line_no, item_id=item_id, message=f"annotation[{ann_idx}][1]은 문자열이어야 합니다.", value=repr(type(sentiment)), sentence_form=sentence_form, annotation=annotation)
                    continue

                aspect_counts[aspect] += 1
                sentiment_counts[sentiment] += 1

                if aspect not in VALID_ASPECTS:
                    add_issue(issues, issue_type="aspect_out_of_set", line_no=line_no, item_id=item_id, message=f"annotation[{ann_idx}]에 정의되지 않은 속성이 있습니다.", value=aspect, sentence_form=sentence_form, annotation=annotation)

                if sentiment not in VALID_SENTIMENTS:
                    add_issue(issues, issue_type="sentiment_out_of_set", line_no=line_no, item_id=item_id, message=f"annotation[{ann_idx}]에 유효하지 않은 감정이 있습니다.", value=sentiment, sentence_form=sentence_form, annotation=annotation)

                key = (aspect, sentiment)
                if key in seen_in_row:
                    add_issue(issues, issue_type="duplicate_annotation_in_row", line_no=line_no, item_id=item_id, message=f"annotation[{ann_idx}]에 중복 annotation이 있습니다.", value=list(key), sentence_form=sentence_form, annotation=annotation)
                seen_in_row.add(key)

    for item_id, lines in id_to_lines.items():
        if len(lines) > 1:
            add_issue(issues, issue_type="duplicate_id", line_no=lines[0], item_id=item_id, message="같은 ID가 여러 줄에 나타납니다.", value=lines)

    summary = {
        "total_rows": total_nonempty_lines,
        "annotated_rows": annotated_rows,
        "total_annotations": total_annotations,
        "total_issues": len(issues),
        "aspect_out_of_set": sum(1 for x in issues if x["issue_type"] == "aspect_out_of_set"),
        "sentiment_out_of_set": sum(1 for x in issues if x["issue_type"] == "sentiment_out_of_set"),
        "format_issues": sum(1 for x in issues if x["issue_type"] not in {"aspect_out_of_set", "sentiment_out_of_set", "duplicate_id", "duplicate_annotation_in_row"}),
        "duplicate_id_issues": sum(1 for x in issues if x["issue_type"] == "duplicate_id"),
        "duplicate_annotation_issues": sum(1 for x in issues if x["issue_type"] == "duplicate_annotation_in_row"),
    }

    return summary, issues, aspect_counts, sentiment_counts


summary, issues, aspect_counts, sentiment_counts = validate_jsonl(DATA_FILE)
summary

{'total_rows': 12000,
 'annotated_rows': 7748,
 'total_annotations': 13069,
 'total_issues': 6,
 'aspect_out_of_set': 1,
 'sentiment_out_of_set': 0,
 'format_issues': 0,
 'duplicate_id_issues': 0,
 'duplicate_annotation_issues': 5}

In [4]:
issue_counts = Counter(issue["issue_type"] for issue in issues)

if pd is not None:
    display(pd.DataFrame([summary]))
    display(pd.DataFrame(issue_counts.most_common(), columns=["문제 종류", "건수"]))
else:
    print("==== 검증 요약 ====")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("\n==== 문제 종류별 건수 ====")
    print(json.dumps(dict(issue_counts), ensure_ascii=False, indent=2))

,total_rows,annotated_rows,total_annotations,total_issues,aspect_out_of_set,sentiment_out_of_set,format_issues,duplicate_id_issues,duplicate_annotation_issues
0,12000,7748,13069,6,1,0,0,0,5


,문제 종류,건수
0,duplicate_annotation_in_row,5
1,aspect_out_of_set,1


In [5]:
bad_aspects = [issue for issue in issues if issue["issue_type"] == "aspect_out_of_set"]
bad_sentiments = [issue for issue in issues if issue["issue_type"] == "sentiment_out_of_set"]
format_issues = [
    issue for issue in issues
    if issue["issue_type"] not in {"aspect_out_of_set", "sentiment_out_of_set", "duplicate_id", "duplicate_annotation_in_row"}
]

print(f"정의되지 않은 속성: {len(bad_aspects)}건")
print(f"유효하지 않은 감정: {len(bad_sentiments)}건")
print(f"포맷 오류: {len(format_issues)}건")

if pd is not None:
    if bad_aspects:
        display(pd.DataFrame(bad_aspects))
    if bad_sentiments:
        display(pd.DataFrame(bad_sentiments))
    if format_issues:
        display(pd.DataFrame(format_issues))
else:
    print(json.dumps({"bad_aspects": bad_aspects, "bad_sentiments": bad_sentiments, "format_issues": format_issues}, ensure_ascii=False, indent=2))

정의되지 않은 속성: 1건
유효하지 않은 감정: 0건
포맷 오류: 0건


,issue_type,line_no,id,message,value,sentence_form,annotation
0,aspect_out_of_set,8379,steam_08382,annotation[9]에 정의되지 않은 속성이 있습니다.,시스템#자유度,✅1. 장르 및 플레이 스타일\n장르: 하이퍼 fps\n\n시점: 3인칭 \n\n플...,"[[콘텐츠#볼륨, positive], [UX#사운드, positive], [스토리#..."


In [6]:
invalid_aspect_counts = Counter(issue["value"] for issue in bad_aspects)
invalid_sentiment_counts = Counter(issue["value"] for issue in bad_sentiments)

if pd is not None:
    display(pd.DataFrame(invalid_aspect_counts.most_common(), columns=["정의되지 않은 속성", "건수"]))
    display(pd.DataFrame(invalid_sentiment_counts.most_common(), columns=["유효하지 않은 감정", "건수"]))
    display(pd.DataFrame(aspect_counts.most_common(), columns=["속성명", "건수"]))
    display(pd.DataFrame(sentiment_counts.most_common(), columns=["감정", "건수"]))
else:
    print("==== 정의되지 않은 속성 ====")
    print(json.dumps(dict(invalid_aspect_counts), ensure_ascii=False, indent=2))
    print("\n==== 유효하지 않은 감정 ====")
    print(json.dumps(dict(invalid_sentiment_counts), ensure_ascii=False, indent=2))
    print("\n==== 속성명 분포 ====")
    print(json.dumps(dict(aspect_counts), ensure_ascii=False, indent=2))
    print("\n==== 감정 분포 ====")
    print(json.dumps(dict(sentiment_counts), ensure_ascii=False, indent=2))

,정의되지 않은 속성,건수
0,시스템#자유度,1


,유효하지 않은 감정,건수


,속성명,건수
0,콘텐츠#몰입도,5178
1,시스템#밸런스,910
2,시스템#진입장벽,821
3,운영#업데이트,741
4,콘텐츠#볼륨,705
5,콘텐츠#난이도,701
6,시스템#독창성,563
7,시스템#자유도,485
8,콘텐츠#피로도,483
9,시스템#버그,429


,감정,건수
0,positive,7180
1,negative,5181
2,neutral,708


In [7]:
# 옵션: 상세 문제 사항을 파일로 저장
SAVE_ISSUES = False
ISSUE_FILE = Path("analysis_output/steam_train_12k_validation_issues.jsonl")

if SAVE_ISSUES:
    ISSUE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with ISSUE_FILE.open("w", encoding="utf-8") as f:
        for issue in issues:
            f.write(json.dumps(issue, ensure_ascii=False) + "\n")
    print(f"{len(issues)}개의 문제 사항을 {ISSUE_FILE}에 저장했습니다.")
else:
    print("SAVE_ISSUES가 False입니다. 상세 내용을 저장하려면 True로 변경하세요.")

SAVE_ISSUES가 False입니다. 상세 내용을 저장하려면 True로 변경하세요.


## 상세 문제 사항 확인

In [ ]:
print("=" * 80)
print("검증 결과: 정의되지 않은 속성")
print("=" * 80)

if bad_aspects:
    for issue in bad_aspects:
        print(f"\n줄 번호: {issue['line_no']}")
        print(f"ID: {issue['id']}")
        print(f"잘못된 속성: '{issue['value']}'")
        print(f"리뷰 텍스트: {issue['sentence_form'][:100]}")
        print(f"전체 annotation:")
        for anno in issue['annotation']:
            print(f"   - {anno}")
else:
    print("정의되지 않은 속성 없음")

print("\n" + "=" * 80)
print("검증 결과: 중복 annotation")
print("=" * 80)

duplicate_issues = [issue for issue in issues if issue["issue_type"] == "duplicate_annotation_in_row"]

if duplicate_issues:
    for issue in duplicate_issues:
        print(f"\n줄 번호: {issue['line_no']}")
        print(f"ID: {issue['id']}")
        print(f"중복 annotation: {issue['value']}")
        print(f"리뷰 텍스트: {issue['sentence_form'][:100]}")
else:
    print("중복 annotation 없음")

## 최종 검증 결과

In [ ]:
print("\n" + "=" * 80)
print("Part 1 검증 결과 요약")
print("=" * 80)

print(f"\n데이터 규모:")
print(f"   총 리뷰 수: {summary['total_rows']:,}개")
print(f"   라벨이 있는 리뷰: {summary['annotated_rows']:,}개 ({100*summary['annotated_rows']/summary['total_rows']:.1f}%)")
print(f"   총 annotation: {summary['total_annotations']:,}개")
print(f"   리뷰당 평균 annotation: {summary['total_annotations']/summary['annotated_rows']:.2f}개")

print(f"\n발견된 문제:")
problem_list = []

if summary['aspect_out_of_set'] > 0:
    print(f"   - 정의되지 않은 속성: {summary['aspect_out_of_set']}건")
    problem_list.append(f"정의되지 않은 속성 {summary['aspect_out_of_set']}건")
else:
    print(f"   - 정의된 속성 범위 내 (통과)")

if summary['sentiment_out_of_set'] > 0:
    print(f"   - 유효하지 않은 감정: {summary['sentiment_out_of_set']}건")
    problem_list.append(f"유효하지 않은 감정 {summary['sentiment_out_of_set']}건")
else:
    print(f"   - 모든 감정이 유효 (통과)")

if summary['duplicate_annotation_issues'] > 0:
    print(f"   - 중복 annotation: {summary['duplicate_annotation_issues']}건")
    problem_list.append(f"중복 annotation {summary['duplicate_annotation_issues']}건")
else:
    print(f"   - 중복 annotation 없음 (통과)")

if summary['duplicate_id_issues'] > 0:
    print(f"   - 중복 ID: {summary['duplicate_id_issues']}건")
    problem_list.append(f"중복 ID {summary['duplicate_id_issues']}건")
else:
    print(f"   - 중복 ID 없음 (통과)")

print(f"\n발견된 문제 수: {len(problem_list)}건")
if problem_list:
    print("필요한 수정 사항:")
    for prob in problem_list:
        print(f"   - {prob}")

print("=" * 80)

## Part 2: 데이터 전처리 및 결과

Part 1에서 발견된 6개의 문제를 수정합니다.

**수정 규칙:**
1. **속성명 오류 수정**: 시스템#자유度 → 시스템#자유도
2. **중복 annotation 제거**: 같은 속성-감정 조합이 중복되면 1개만 남김
3. **감정 충돌 처리**: 같은 속성에 여러 감정이 있으면 빈도가 높은 감정 선택, 같으면 neutral 선택

In [ ]:
def fix_annotations(annotation_list):
    """
    Annotation 수정:
    1. 시스템#자유度 → 시스템#자유도
    2. 같은 속성인데 다른 감정이면:
       - 개수가 많은 감정을 선택 (예: positive 2개 + negative 1개 → positive)
       - 개수가 같으면 neutral 선택 (예: positive 1개 + negative 1개 → neutral)
    3. 같은 속성+감정 중복은 1개로 줄임
    """
    if not annotation_list:
        return []

    # Step 1: 속성명 오류 수정
    fixed = []
    for anno in annotation_list:
        aspect, sentiment = anno[0], anno[1]
        # 시스템#자유度 → 시스템#자유도
        if aspect == "시스템#자유度":
            aspect = "시스템#자유도"
        fixed.append([aspect, sentiment])

    # Step 2: 같은 속성별로 모든 감정 수집 (중복 포함하여 개수 파악)
    aspect_sentiments = defaultdict(list)
    for aspect, sentiment in fixed:
        aspect_sentiments[aspect].append(sentiment)

    result = []

    for aspect, sentiments in aspect_sentiments.items():
        unique_sentiments = set(sentiments)

        if len(unique_sentiments) == 1:
            # 이 속성에 감정이 1개만 있으면 그대로 사용
            result.append([aspect, sentiments[0]])
        else:
            # 같은 속성에 여러 감정이 있으면, 개수가 많은 것을 선택
            sentiment_counts = Counter(sentiments)
            max_count = max(sentiment_counts.values())

            # 최대 개수인 감정들을 찾기
            candidates = [s for s, c in sentiment_counts.items() if c == max_count]

            if len(candidates) == 1:
                # 하나만 있으면 그것 선택 (예: positive 2개 + negative 1개 → positive)
                chosen_sentiment = candidates[0]
            else:
                # 개수가 같으면 neutral 선택 (예: positive 1개 + negative 1개 → neutral)
                chosen_sentiment = "neutral"

            result.append([aspect, chosen_sentiment])

    return result


📝 수정 함수 테스트:

Test 1:
  입력:  [['시스템#자유度', 'positive'], ['시스템#자유度', 'positive']]
  출력:  [['시스템#자유도', 'positive']]

Test 2:
  입력:  [['콘텐츠#몰입도', 'positive'], ['콘텐츠#몰입도', 'positive'], ['콘텐츠#몰입도', 'negative']]
  출력:  [['콘텐츠#몰입도', 'positive']]

Test 3:
  입력:  [['콘텐츠#볼륨', 'positive'], ['콘텐츠#볼륨', 'positive'], ['콘텐츠#볼륨', 'negative']]
  출력:  [['콘텐츠#볼륨', 'positive']]

Test 4:
  입력:  [['UX#그래픽', 'positive'], ['UX#그래픽', 'negative']]
  출력:  [['UX#그래픽', 'neutral']]



In [ ]:
# 전체 데이터셋 수정 및 저장
from pathlib import Path

INPUT_FILE = Path("labelled_AI_final/steam_train_12k.jsonl")
OUTPUT_FILE = Path("labelled_AI_final/steam_train_12k_cleaned.jsonl")

stats = {
    "total_rows": 0,
    "rows_with_fixes": 0,
    "attribute_fixes": 0,
    "duplicate_fixes": 0,
    "conflict_fixes": 0,
    "total_annotations_before": 0,
    "total_annotations_after": 0,
}

cleaned_data = []

with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line.strip())
        stats["total_rows"] += 1
        
        if record["annotation"]:
            stats["total_annotations_before"] += len(record["annotation"])
            
            original_annotation = record["annotation"].copy()
            fixed_annotation = fix_annotations(record["annotation"])
            
            stats["total_annotations_after"] += len(fixed_annotation)
            
            if fixed_annotation != original_annotation:
                stats["rows_with_fixes"] += 1
                
                original_set = set(tuple(a) for a in original_annotation)
                fixed_set = set(tuple(a) for a in fixed_annotation)
                
                for orig_anno in original_annotation:
                    if orig_anno[0] == "시스템#자유度":
                        stats["attribute_fixes"] += 1
                
                if len(original_annotation) > len(fixed_annotation):
                    stats["duplicate_fixes"] += len(original_annotation) - len(fixed_annotation)
                
                for fixed_anno in fixed_annotation:
                    aspect = fixed_anno[0]
                    original_sentiments = [a[1] for a in original_annotation if a[0] == aspect]
                    if len(set(original_sentiments)) > 1:
                        stats["conflict_fixes"] += 1
                        break
            
            record["annotation"] = fixed_annotation
        else:
            stats["total_annotations_before"] += 0
            stats["total_annotations_after"] += 0
        
        cleaned_data.append(record)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for record in cleaned_data:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("=" * 80)
print("데이터 전처리 완료")
print("=" * 80)
print(f"\n수정 통계:")
print(f"   총 리뷰 수: {stats['total_rows']:,}개")
print(f"   수정된 리뷰: {stats['rows_with_fixes']:,}개 ({100*stats['rows_with_fixes']/stats['total_rows']:.2f}%)")
print(f"   속성명 수정: {stats['attribute_fixes']}건")
print(f"   중복 제거: {stats['duplicate_fixes']}건")
print(f"   감정 충돌 처리: {stats['conflict_fixes']}건")

print(f"\nAnnotation 변화:")
print(f"   수정 전: {stats['total_annotations_before']:,}개")
print(f"   수정 후: {stats['total_annotations_after']:,}개")
print(f"   감소: {stats['total_annotations_before'] - stats['total_annotations_after']}개 ({100*(stats['total_annotations_before']-stats['total_annotations_after'])/stats['total_annotations_before']:.2f}%)")

print(f"\n저장 위치: {OUTPUT_FILE.resolve()}")
print("=" * 80)

## 수정된 파일 검증

In [ ]:
# 수정된 파일 검증
summary_cleaned, issues_cleaned, aspect_counts_cleaned, sentiment_counts_cleaned = validate_jsonl(OUTPUT_FILE)

print("\n" + "=" * 80)
print("전처리 후 검증 결과")
print("=" * 80)

print(f"\n데이터 규모:")
print(f"   총 리뷰 수: {summary_cleaned['total_rows']:,}개")
print(f"   라벨이 있는 리뷰: {summary_cleaned['annotated_rows']:,}개 ({100*summary_cleaned['annotated_rows']/summary_cleaned['total_rows']:.1f}%)")
print(f"   총 annotation: {summary_cleaned['total_annotations']:,}개")
print(f"   리뷰당 평균 annotation: {summary_cleaned['total_annotations']/summary_cleaned['annotated_rows']:.2f}개")

print(f"\n검증 결과:")
remaining_issues = 0

if summary_cleaned['aspect_out_of_set'] > 0:
    print(f"   - 정의되지 않은 속성: {summary_cleaned['aspect_out_of_set']}건")
    remaining_issues += 1
else:
    print(f"   - 정의된 속성 범위 내 (통과)")

if summary_cleaned['sentiment_out_of_set'] > 0:
    print(f"   - 유효하지 않은 감정: {summary_cleaned['sentiment_out_of_set']}건")
    remaining_issues += 1
else:
    print(f"   - 모든 감정이 유효 (통과)")

if summary_cleaned['duplicate_annotation_issues'] > 0:
    print(f"   - 중복 annotation: {summary_cleaned['duplicate_annotation_issues']}건")
    remaining_issues += 1
else:
    print(f"   - 중복 annotation 없음 (통과)")

if summary_cleaned['duplicate_id_issues'] > 0:
    print(f"   - 중복 ID: {summary_cleaned['duplicate_id_issues']}건")
    remaining_issues += 1
else:
    print(f"   - 중복 ID 없음 (통과)")

print(f"\n결론:")
if remaining_issues == 0:
    print(f"   전처리 완료 - 모든 검증 통과")
else:
    print(f"   {remaining_issues}가지 문제 남음")

print("=" * 80)

## 전처리 결과 비교

In [ ]:
print("\n" + "=" * 80)
print("수정 전후 비교")
print("=" * 80)

comparison = {
    "지표": ["총 annotation", "평균 annotation/리뷰", "정의되지 않은 속성", "중복 annotation"],
    "수정 전": [
        summary['total_annotations'],
        f"{summary['total_annotations']/summary['annotated_rows']:.2f}",
        summary['aspect_out_of_set'],
        summary['duplicate_annotation_issues']
    ],
    "수정 후": [
        summary_cleaned['total_annotations'],
        f"{summary_cleaned['total_annotations']/summary_cleaned['annotated_rows']:.2f}",
        summary_cleaned['aspect_out_of_set'],
        summary_cleaned['duplicate_annotation_issues']
    ]
}

if pd is not None:
    comparison_df = pd.DataFrame(comparison)
    display(comparison_df)
else:
    for i in range(len(comparison["지표"])):
        print(f"\n{comparison['지표'][i]}:")
        print(f"  수정 전: {comparison['수정 전'][i]}")
        print(f"  수정 후: {comparison['수정 후'][i]}")

print("\n" + "=" * 80)
print("속성 분포 (상위 10개)")
print("=" * 80)

if pd is not None:
    before_df = pd.DataFrame(aspect_counts.most_common(10), columns=["속성", "수정 전"])
    after_df = pd.DataFrame(aspect_counts_cleaned.most_common(10), columns=["속성", "수정 후"])
    
    merge_df = before_df.set_index("속성").join(after_df.set_index("속성"), how="outer").fillna(0).astype(int)
    merge_df["변화"] = merge_df["수정 후"] - merge_df["수정 전"]
    
    display(merge_df)

print("=" * 80)